# Sequential Decision Problems: the 4×3 World
### Companion notebook to AIMA (4th ed.), Sections 16.1–16.2

This notebook reproduces, step by step, the running example of Chapter 16: the **4×3 grid world**.

| Topic | Book reference |
|---|---|
| The MDP, the Q-VALUE function | §16.1, p. 559 |
| Utilities of states, Bellman equation | Fig. 16.3, Eq. (16.5) |
| Optimal policy and its dependence on $r$ | Fig. 16.2 (a), (b) |
| Finite horizon ⇒ nonstationary policy | §16.1.1 |
| Reward shaping theorem | Eq. (16.9) |
| Value iteration, contraction, error bounds | Fig. 16.6, 16.7, 16.8, Eqs. (16.10)–(16.13) |
| Policy iteration (exact & modified) | Fig. 16.9, Eq. (16.14) |
| Linear programming | §16.2.3 |
| Online: expectimax, $\epsilon$-horizon, sampling, UCT | §16.2.4, Fig. 16.11 |

**The environment.** States are squares $(\text{row}, \text{column})$, row $\in\{1,2,3\}$ counted **from the bottom** (as in the book's figures), column $\in\{1,..,4\}$; $(2,2)$ is a wall. $(3,4)$ and $(2,4)$ are terminal.

> **Notation.** The book writes squares as $(x,y)$ = (column, row). This notebook uses (row, column), so the book's $(4,3)$ is our $(3,4)$, the book's $(3,1)$ is our $(1,3)$, and so on: just swap the two numbers. Each action (Up, Down, Left, Right) moves in the intended direction with probability 0.8 and at right angles with probability 0.1 each; bumping into a wall or the edge leaves the agent in place. The reward is $r=-0.04$ for each transition into a non-terminal state and $+1$ / $-1$ for the transition into $(3,4)$ / $(2,4)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

## 1. The MDP

We store the model as dense arrays so that every algorithm becomes a few lines of linear algebra:

* `P[s, a, s']` $= P(s'\mid s,a)$
* `R[s, a, s']` $= R(s,a,s')$

Terminal states are absorbing with no reward, so their utility is always $0$.

In [ ]:
class GridMDP:
    ACTIONS = ['Up', 'Down', 'Left', 'Right']
    ARROW   = {'Up': '↑', 'Down': '↓', 'Left': '←', 'Right': '→'}
    # states are (row, col); row 1 is the BOTTOM row, so Up increases the row
    DELTA   = {'Up': (1, 0), 'Down': (-1, 0), 'Left': (0, -1), 'Right': (0, 1)}
    PERP    = {'Up': ('Left', 'Right'), 'Down': ('Left', 'Right'),
               'Left': ('Up', 'Down'), 'Right': ('Up', 'Down')}

    def __init__(self, r=-0.04, gamma=1.0, width=4, height=3,
                 walls=((2, 2),), terminals={(3, 4): +1.0, (2, 4): -1.0},
                 p_intended=0.8):
        self.r, self.gamma, self.W, self.H = r, gamma, width, height
        self.walls, self.terminals = set(walls), dict(terminals)
        self.states = [(row, col) for row in range(1, height + 1) for col in range(1, width + 1)
                       if (row, col) not in self.walls]
        self.idx = {s: i for i, s in enumerate(self.states)}
        self.nS, self.nA = len(self.states), len(self.ACTIONS)
        self.is_terminal = np.array([s in self.terminals for s in self.states])
        p_side = (1 - p_intended) / 2

        self.P = np.zeros((self.nS, self.nA, self.nS))
        for s in self.states:
            i = self.idx[s]
            if s in self.terminals:            # absorbing, no further reward
                self.P[i, :, i] = 1.0
                continue
            for a_i, a in enumerate(self.ACTIONS):
                for actual, p in [(a, p_intended), (self.PERP[a][0], p_side), (self.PERP[a][1], p_side)]:
                    self.P[i, a_i, self.idx[self._move(s, actual)]] += p

        # R(s, a, s') depends only on s' here (and is zero out of terminal states)
        reward_in = np.array([self.terminals.get(s, r) for s in self.states])
        self.R = np.where(self.is_terminal[:, None, None], 0.0, reward_in[None, None, :]) * np.ones_like(self.P)

    def _move(self, s, a):
        row, col = s[0] + self.DELTA[a][0], s[1] + self.DELTA[a][1]
        if not (1 <= row <= self.H and 1 <= col <= self.W) or (row, col) in self.walls:
            return s
        return (row, col)

    # ---- the book's Q-VALUE(mdp, s, a, U), vectorised over all (s, a) ----
    def q_values(self, U):
        return np.einsum('ijk,ijk->ij', self.P, self.R + self.gamma * U[None, None, :])

    def sample(self, s_i, a_i, rng=rng):
        s2 = rng.choice(self.nS, p=self.P[s_i, a_i])
        return s2, self.R[s_i, a_i, s2]

mdp = GridMDP()
print(mdp.nS, 'states (row, col):', mdp.states)

### Plotting helpers

In [ ]:
def draw_grid(mdp, values=None, policy=None, ax=None, title='', fmt='{:.4f}', ties=None):
    """values: array over states; policy: array of action indices (or None);
    ties: optional list of sets of optimal actions per state (draws all of them).
    State (row, col) is drawn at x = col, y = row (row 1 at the bottom)."""
    ax = ax or plt.gca()
    for row, col in product(range(1, mdp.H + 1), range(1, mdp.W + 1)):
        s = (row, col)
        color = '#444' if s in mdp.walls else ('#cfe8cf' if mdp.terminals.get(s, 0) > 0
                 else '#f4c7c3' if s in mdp.terminals else 'white')
        ax.add_patch(plt.Rectangle((col - .5, row - .5), 1, 1, fc=color, ec='k'))
        if s in mdp.walls:
            continue
        i = mdp.idx[s]
        if s in mdp.terminals:
            ax.text(col, row, f'{mdp.terminals[s]:+g}', ha='center', va='center', fontsize=16, weight='bold')
            continue
        if policy is not None or ties is not None:
            acts = ties[i] if ties is not None else {policy[i]}
            ax.text(col, row + .12, ''.join(mdp.ARROW[mdp.ACTIONS[a]] for a in sorted(acts)),
                    ha='center', va='center', fontsize=18)
        if values is not None:
            ax.text(col, row - .25, fmt.format(values[i]), ha='center', va='center', fontsize=9)
    ax.set_xlim(.5, mdp.W + .5); ax.set_ylim(.5, mdp.H + .5); ax.set_aspect('equal')
    ax.set_xticks(range(1, mdp.W + 1)); ax.set_yticks(range(1, mdp.H + 1))
    ax.set_xlabel('column'); ax.set_ylabel('row')
    ax.set_title(title)
    return ax

### The MDP as a graph

Drawing the whole MDP as one graph (11 states × 4 actions × up to 3 outcomes each) is too crowded to read. Instead we draw:

* **one state at a time**: the state → its 4 actions → the possible next states, with $P(s'\mid s,a)$ on the edges and $R$ under each next state. This is one level of an expectimax tree, and with utilities $U$ it shows exactly the Q-VALUE computation $Q(s,a)=\sum_{s'}P(s'\mid s,a)[R+\gamma U(s')]$;
* **a fixed policy on the grid**: once $\pi$ is chosen, the MDP becomes a Markov chain; arrows show where the agent can go from each square, thicker = more likely.

In [ ]:
def _node(ax, xy, text, fc='white', r=0.42, fontsize=9, ec='k', lw=1.2):
    ax.add_patch(plt.Circle(xy, r, fc=fc, ec=ec, lw=lw, zorder=3))
    ax.text(*xy, text, ha='center', va='center', fontsize=fontsize, zorder=4)

def draw_state_graph(mdp, s, U=None, highlight=None, ax=None, title=None):
    """One-step look-ahead from state s = (row, col).
    U: optional utilities -> shown in the next-state nodes and used to print Q(s, a) at each action.
    highlight: optional set of action indices to draw in bold (e.g. the optimal ones)."""
    ax = ax or plt.gca()
    i = mdp.idx[s]
    succ = sorted({j for a in range(mdp.nA) for j in np.flatnonzero(mdp.P[i, a])},
                  key=lambda j: (-mdp.states[j][0], mdp.states[j][1]))      # top rows first
    y_act = np.linspace(1.5, -1.5, mdp.nA)
    y_succ = {j: y for j, y in zip(succ, np.linspace(1.5, -1.5, len(succ)) if len(succ) > 1 else [0])}
    Q = mdp.q_values(U) if U is not None else None
    hl = set(highlight) if highlight is not None else set()

    # state node
    lab = f'{s}' + (f'\nU={U[i]:.3f}' if U is not None else '')
    _node(ax, (0, 0), lab, fc='#dde8f7', r=0.55)

    for a in range(mdp.nA):
        bold = a in hl
        col, lw = ('#c0392b', 2.6) if bold else ('#555', 1.0)
        ya = y_act[a]
        ax.plot([0.55, 1.75], [0, ya], color=col, lw=lw, zorder=1)
        # action node (small, like a chance node in expectimax)
        ax.add_patch(plt.Circle((2, ya), 0.25, fc='#fff3c4' if bold else '#f2f2f2', ec=col, lw=lw, zorder=3))
        ax.text(2, ya, mdp.ARROW[mdp.ACTIONS[a]], ha='center', va='center', fontsize=13, zorder=4)
        if Q is not None:
            ax.text(2, ya + 0.38, f'Q={Q[i, a]:.3f}', ha='center', va='center', fontsize=8, color=col)
        for j in np.flatnonzero(mdp.P[i, a]):
            p = mdp.P[i, a, j]
            x0, y0, x1, y1 = 2.25, ya, 4.6, y_succ[j]
            ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                        arrowprops=dict(arrowstyle='->', color=col, lw=lw * (0.6 + p), shrinkA=0, shrinkB=0),
                        zorder=2)
            t = 0.2                                         # label close to its own action node
            ax.text(x0 + t * (x1 - x0), y0 + t * (y1 - y0), f'{p:g}', fontsize=7, color=col,
                    ha='center', va='center', bbox=dict(fc='white', ec='none', pad=0.5), zorder=5)

    for j in succ:
        s2 = mdp.states[j]
        fc = '#cfe8cf' if mdp.terminals.get(s2, 0) > 0 else '#f4c7c3' if s2 in mdp.terminals else 'white'
        lab = f'{s2}' + (' (stay)' if j == i else '')
        if U is not None:
            lab += f'\nU={U[j]:.3f}'
        _node(ax, (5.05, y_succ[j]), lab, fc=fc, r=0.45, fontsize=8)
        ax.text(5.6, y_succ[j], f'R={mdp.R[i, 0, j]:+g}', va='center', fontsize=8, color='#333')

    ax.set_xlim(-0.8, 6.4); ax.set_ylim(-2.1, 2.1); ax.set_aspect('equal'); ax.axis('off')
    ax.set_title(title or f'State {s}: actions and outcomes')
    return ax

def draw_policy_graph(mdp, pi, U=None, ax=None, title=''):
    """The Markov chain obtained by fixing policy pi, drawn on the grid.
    Arrow width ~ transition probability; the number in the corner of a square = probability of staying."""
    ax = draw_grid(mdp, values=U, policy=pi, ax=ax, title=title)
    for i, s in enumerate(mdp.states):
        if mdp.is_terminal[i]:
            continue
        for j in np.flatnonzero(mdp.P[i, pi[i]]):
            p = mdp.P[i, pi[i], j]
            if j == i:
                ax.text(s[1] + .42, s[0] + .40, f'{p:g}', ha='right', va='top', fontsize=7, color='#1f5fa8')
                continue
            s2 = mdp.states[j]
            dy, dx = s2[0] - s[0], s2[1] - s[1]
            ox, oy = -dy * 0.12, dx * 0.12          # shift sideways so A->B and B->A don't overlap
            ax.annotate('', xy=(s2[1] - .38 * dx + ox, s2[0] - .38 * dy + oy),
                        xytext=(s[1] + .38 * dx + ox, s[0] + .38 * dy + oy),
                        arrowprops=dict(arrowstyle='-|>', color='#1f5fa8', lw=0.5 + 3 * p, alpha=.85))
    return ax

# The book discusses state (3,1) in its notation = our (1,3): Up passes the -1, Left takes the long way round
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
draw_state_graph(mdp, (1, 1), ax=axes[0])
draw_state_graph(mdp, (1, 3), ax=axes[1])
plt.tight_layout(); plt.show()

## 2. Utilities of states and the Bellman equation (Fig. 16.3)

The utility of a state is the expected discounted return under an optimal policy; it satisfies the **Bellman equation** (16.5)

$$U(s)=\max_{a\in A(s)}\sum_{s'}P(s'\mid s,a)\,[R(s,a,s')+\gamma U(s')].$$

We solve it with **value iteration** (Fig. 16.6): start from $U_0=0$ and repeat the Bellman update (16.10)
$U_{i+1}\leftarrow BU_i$ until the update is smaller than $\epsilon(1-\gamma)/\gamma$ (Eq. 16.12).
Because $\gamma=1$ in Fig. 16.3, that criterion degenerates to $0$; we then simply stop when the change falls below a small tolerance (the remark at the end of §16.2.1: with terminal states and proper policies, convergence still holds).

In [ ]:
def value_iteration(mdp, eps=1e-6, max_iter=100_000, history=False):
    U = np.zeros(mdp.nS)
    hist = [U.copy()]
    tol = eps * (1 - mdp.gamma) / mdp.gamma if mdp.gamma < 1 else eps
    for _ in range(max_iter):
        U_new = mdp.q_values(U).max(axis=1)
        U_new[mdp.is_terminal] = 0.0
        delta = np.abs(U_new - U).max()
        U = U_new
        hist.append(U.copy())
        if delta <= tol:
            break
    return (U, np.array(hist)) if history else U

def greedy_policy(mdp, U, tie_tol=1e-4):
    '''Eq. (16.4): pi*(s) = argmax_a sum_s' P(s'|s,a)[R + gamma U(s')]. Also returns the set of tied actions.'''
    Q = mdp.q_values(U)
    pi = Q.argmax(axis=1)
    ties = [set(np.flatnonzero(Q[i] >= Q[i].max() - tie_tol)) for i in range(mdp.nS)]
    return pi, ties

U_star = value_iteration(mdp, eps=1e-12)
pi_star, ties = greedy_policy(mdp, U_star)

# the book's Fig. 16.3, converted to (row, col)
book = {(3,1):.8516,(3,2):.9078,(3,3):.9578,(2,1):.8016,(2,3):.7003,(1,1):.7453,(1,2):.6953,(1,3):.6514,(1,4):.4279}
for s, v in book.items():
    print(f'{s}: computed {U_star[mdp.idx[s]]:.4f}   book {v:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
draw_grid(mdp, values=U_star, ax=axes[0], title='Utilities U(s), γ=1, r=−0.04  (Fig. 16.3)')
draw_grid(mdp, ties=ties, ax=axes[1], fmt='', title='Optimal policy (Fig. 16.2a)')
plt.tight_layout(); plt.show()

### The same graphs with the utilities filled in

Left: the one-step look-ahead from $(1,3)$ with $U^*$ in every node. Each action's $Q$ is the probability-weighted sum along its edges, and $U(1,3)=\max_a Q$; the optimal action is drawn in red.
Right: the Markov chain induced by the optimal policy.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4.8), gridspec_kw={'width_ratios': [1.25, 1]})
s = (1, 3)
draw_state_graph(mdp, s, U=U_star, highlight=ties[mdp.idx[s]], ax=axes[0],
                 title=f'State {s}: Q(s,a) = Σ P(s′|s,a)[R + γU(s′)]')
draw_policy_graph(mdp, pi_star, U=U_star, ax=axes[1], title='Markov chain under the optimal policy')
plt.tight_layout(); plt.show()

### Is there really a tie in $(1,3)$?

The caption of Fig. 16.2(a) says Left (the long, safe route) and Up (the short route past the $-1$) are both optimal in $(1,3)$ (the book's $(3,1)$). With the model as specified — and the book's own utilities from Fig. 16.3 — the two are close but **not** equal: Left is strictly better. Let's look at the Q-values and find, by bisection, the value of $r$ at which they *do* tie.

In [ ]:
q13 = mdp.q_values(U_star)[mdp.idx[(1, 3)]]
print('Q((1,3), a) at r = -0.04:', dict(zip(mdp.ACTIONS, q13.round(4))))

def up_minus_left(r):
    m = GridMDP(r=r); U = value_iteration(m, 1e-13, max_iter=10**6)
    q = m.q_values(U)[m.idx[(1, 3)]]
    return q[m.ACTIONS.index('Up')] - q[m.ACTIONS.index('Left')]

lo, hi = -0.06, -0.03            # Up is better at lo, Left at hi
for _ in range(40):
    mid = (lo + hi) / 2
    lo, hi = (mid, hi) if up_minus_left(mid) > 0 else (lo, mid)
print(f'Up and Left tie in (1,3) at r ≈ {lo:.5f}')

### Checking one Bellman equation by hand: $U(1,1)$

The book writes out the four terms (Up, Left, Down, Right) for $(1,1)$; plugging in the utilities shows Up wins.

In [ ]:
U = lambda row, col: U_star[mdp.idx[(row, col)]]
g, r = 1.0, -0.04
# from (1,1): Up -> (2,1), Right -> (1,2), Left/Down bump into the edge and stay in (1,1)
terms = {
    'Up':    0.8*(r + g*U(2,1)) + 0.1*(r + g*U(1,2)) + 0.1*(r + g*U(1,1)),
    'Left':  0.9*(r + g*U(1,1)) + 0.1*(r + g*U(2,1)),
    'Down':  0.9*(r + g*U(1,1)) + 0.1*(r + g*U(1,2)),
    'Right': 0.8*(r + g*U(1,2)) + 0.1*(r + g*U(2,1)) + 0.1*(r + g*U(1,1)),
}
for a, v in terms.items():
    print(f'{a:6s} {v:.4f}')
print('max =', max(terms.values()), ' vs U(1,1) =', U(1,1))
# Same thing through the generic Q-VALUE function (Eq. 16.8): U(s) = max_a Q(s,a)
print('Q(1,1,·) =', dict(zip(mdp.ACTIONS, mdp.q_values(U_star)[mdp.idx[(1,1)]].round(4))))

## 3. How the optimal policy depends on $r$ (Fig. 16.2b)

Sweeping the per-step reward $r$ shows four qualitatively different regimes:

* $r$ very negative — life is so painful the agent heads for the *nearest* exit, even the $-1$.
* moderately negative — take the shortest route to $+1$, accepting some risk.
* slightly negative — take the long, safe route; never risk the $-1$.
* $r>0$ — life is pleasant: avoid **both** exits forever. This policy is **improper** (never reaches a terminal state); with $\gamma=1$ its utility is $+\infty$, so for that case we use $\gamma=0.99$.

We detect the thresholds numerically by a fine sweep.

In [ ]:
def opt_policy(r, gamma=1.0):
    m = GridMDP(r=r, gamma=gamma)
    U = value_iteration(m, eps=1e-10, max_iter=200_000)
    return m, U, greedy_policy(m, U, tie_tol=1e-9)[0]

rs = np.round(np.linspace(-2.0, -0.002, 500), 4)
prev, changes = None, []
for r in rs:
    _, _, pi = opt_policy(r)
    key = tuple(pi)
    if prev is not None and key != prev:
        changes.append(r)
    prev = key
print('policy changes near r =', changes)

(Changes at closely spaced $r$ values are individual states switching one at a time; with the book's tie-breaking they're grouped into the four ranges of Fig. 16.2(b).)

In [ ]:
examples = [(-2.0, 1.0), (-0.3, 1.0), (-0.01, 1.0), (+0.04, 0.99)]
fig, axes = plt.subplots(1, 4, figsize=(18, 3.6))
for ax, (r, g) in zip(axes, examples):
    m, U, pi = opt_policy(r, g)
    draw_grid(m, policy=pi, ax=ax, title=f'r = {r:+g}, γ = {g}')
plt.tight_layout(); plt.show()

## 4. Finite horizon ⇒ nonstationary policy (§16.1.1)

With a fixed deadline $N$, the optimal action depends on the time left. Backward induction computes $U_k$ = utility with $k$ steps to go, and $\pi_k(s)$ = best first action with $k$ steps to go (this is exactly value iteration, read as "$k$ steps remaining").

The book's example: starting at $(1,3)$ (the book's $(3,1)$) with $N=3$, the agent must go **Up** to have any chance of reaching $+1$; with $N=100$ it can afford the safe route **Left**.

In [ ]:
def finite_horizon(mdp, N):
    U = np.zeros(mdp.nS)
    policies = []                    # policies[k-1] = best action with k steps to go
    for k in range(1, N + 1):
        Q = mdp.q_values(U)
        policies.append(Q.argmax(axis=1))
        U = Q.max(axis=1); U[mdp.is_terminal] = 0
    return U, policies

_, pols = finite_horizon(mdp, 100)
s13 = mdp.idx[(1, 3)]
print('Action in (1,3) with k steps to go:')
for k in [1, 2, 3, 5, 10, 20, 100]:
    print(f'  k = {k:3d}: {mdp.ACTIONS[pols[k-1][s13]]}')
switch = next(k for k in range(1, 101) if mdp.ACTIONS[pols[k-1][s13]] == 'Left')
print(f'The policy in (1,3) switches from Up to Left once at least {switch} steps remain.')

## 5. Why discounting? Bounded utilities and the interest-rate view

* With $|R|\le R_{\max}$ and $\gamma<1$: $U_h\le R_{\max}/(1-\gamma)$ (Eq. 16.1).
* A discount factor $\gamma$ is equivalent to an interest rate $1/\gamma-1$ (e.g. $\gamma=0.9\Rightarrow 11.1\%$).
* Discounting = a probability $1-\gamma$ of "sudden death" at each step. We check this by simulation: an undiscounted episode killed with probability $1-\gamma$ per step has the same expected return as the discounted return.

In [ ]:
for g in [0.5, 0.9, 0.99]:
    print(f'γ={g}: interest rate = {100*(1/g - 1):.1f}%,  bound R_max/(1-γ) = {1/(1-g):.0f}')

def episode_return(mdp, pi, start, gamma_discount=None, kill_prob=0.0, rng=rng, max_steps=10_000):
    s, G, disc = mdp.idx[start], 0.0, 1.0
    for _ in range(max_steps):
        if mdp.is_terminal[s]:
            break
        s, rew = mdp.sample(s, pi[s], rng)
        G += disc * rew
        if rng.random() < kill_prob:      # die *after* collecting this step's reward
            break
        if gamma_discount is not None:
            disc *= gamma_discount
    return G

g = 0.9
m9 = GridMDP(gamma=g); U9 = value_iteration(m9, 1e-10); pi9, _ = greedy_policy(m9, U9)
n = 20_000
disc = np.mean([episode_return(m9, pi9, (1, 1), gamma_discount=g) for _ in range(n)])
kill = np.mean([episode_return(m9, pi9, (1, 1), kill_prob=1-g) for _ in range(n)])
print(f'\nU(1,1) at γ=0.9 (value iteration): {U9[m9.idx[(1,1)]]:.4f}')
print(f'MC discounted return:              {disc:.4f}')
print(f'MC undiscounted + death w.p. 1-γ:  {kill:.4f}')

*Subtlety:* the kill coin must be flipped **after** each transition: the reward of step $t$ is then collected with probability $\gamma^t$ — exactly its discount weight. Flipping it before the transition would shrink every term by an extra factor $\gamma$.

## 6. The reward-shaping theorem (Eq. 16.9)

For **any** potential $\Phi(s)$, replacing $R(s,a,s')$ by $R'(s,a,s')=R(s,a,s')+\gamma\Phi(s')-\Phi(s)$ leaves the optimal policy unchanged, and $U'(s) = U(s)-\Phi(s)$.
With $\Phi=U$, even the **greedy one-step** policy w.r.t. $R'$ is optimal.

(For the potential argument to hold with terminal states, $\Phi$ must be $0$ at the terminals — otherwise the "telescoping sum" leaves a residue at the end of the episode. We use $\gamma=0.9$ here.)

In [ ]:
def shaped(mdp, Phi):
    m = GridMDP(r=mdp.r, gamma=mdp.gamma)
    shaping = mdp.gamma * Phi[None, None, :] - Phi[:, None, None]
    m.R = mdp.R + np.where(mdp.is_terminal[:, None, None], 0.0, shaping)
    return m

base = GridMDP(gamma=0.9)
U_b = value_iteration(base, 1e-12); pi_b, _ = greedy_policy(base, U_b, 1e-9)

for trial in range(3):
    Phi = rng.normal(0, 5, base.nS); Phi[base.is_terminal] = 0
    m2 = shaped(base, Phi)
    U2 = value_iteration(m2, 1e-12); pi2, _ = greedy_policy(m2, U2, 1e-9)
    print(f'random Φ #{trial}: same policy? {np.array_equal(pi_b, pi2)};  '
          f'max|U\' - (U - Φ)| = {np.abs(U2 - (U_b - Phi)).max():.2e}')

# Φ = U: the greedy (myopic) policy w.r.t. R' is already optimal
mU = shaped(base, U_b)
greedy_myopic = np.einsum('ijk,ijk->ij', mU.P, mU.R).argmax(axis=1)
print('\nΦ = U  →  one-step greedy policy is optimal?', np.array_equal(greedy_myopic, pi_b))

## 7. Value iteration in detail (§16.2.1)

### 7.1 Evolution of utilities (Fig. 16.7a)
Information propagates outward from $(3,4)$: distant states first accumulate negative reward, then rise once a path to $+1$ is "discovered".

In [ ]:
_, H = value_iteration(mdp, eps=1e-12, history=True)
plt.figure(figsize=(7, 4))
for s in [(3, 3), (3, 1), (1, 1), (1, 3), (1, 4)]:      # the states plotted in Fig. 16.7(a), as (row, col)
    plt.plot(H[:41, mdp.idx[s]], label=str(s))
plt.xlabel('Number of iterations'); plt.ylabel('Utility estimates')
plt.title('Value iteration on the 4×3 world, γ=1 (Fig. 16.7a)'); plt.legend(); plt.grid(alpha=.3)
plt.show()

### 7.2 The Bellman update is a contraction (Eq. 16.11)

$\|BU-BU'\|_\infty\le\gamma\|U-U'\|_\infty$ for any two utility vectors. Let's test it on random pairs.

In [ ]:
m = GridMDP(gamma=0.9)
def B(U):
    V = m.q_values(U).max(axis=1); V[m.is_terminal] = 0; return V
ratios = []
for _ in range(10_000):
    U1, U2 = rng.normal(0, 10, m.nS), rng.normal(0, 10, m.nS)
    U1[m.is_terminal] = U2[m.is_terminal] = 0
    ratios.append(np.abs(B(U1) - B(U2)).max() / np.abs(U1 - U2).max())
print(f'max ‖BU−BU′‖/‖U−U′‖ over 10,000 random pairs = {max(ratios):.4f}  (γ = {m.gamma})')

### 7.3 Iterations needed vs. $\gamma$ (Fig. 16.7b)

The worst-case bound: $N=\lceil \log(2R_{\max}/\epsilon(1-\gamma))/\log(1/\gamma)\rceil$ with $\epsilon=c\cdot R_{\max}$. $N$ hardly depends on $c$ but explodes as $\gamma\to 1$.

In [ ]:
gam = np.linspace(0.5, 0.999, 300)
plt.figure(figsize=(7, 4))
for c in [1e-4, 1e-3, 1e-2, 1e-1]:
    N = np.ceil(np.log(2 / (c * (1 - gam))) / np.log(1 / gam))
    plt.semilogy(gam, N, label=f'c = {c:g}')
plt.xlabel('Discount factor γ'); plt.ylabel('Iterations required')
plt.title('Worst-case number of value iterations (Fig. 16.7b)'); plt.legend(); plt.grid(alpha=.3, which='both')
plt.show()

### 7.4 Utility error vs. policy loss (Fig. 16.8, Eq. 16.13)

At $\gamma=0.9$ we track, after each iteration $i$,
* the max error $\|U_i-U\|$, and
* the **policy loss** $\|U^{\pi_i}-U\|$, where $\pi_i$ is greedy w.r.t. $U_i$ and $U^{\pi_i}$ is computed exactly by solving the linear system (16.14).

The policy becomes optimal long before the utilities converge.

In [ ]:
def policy_evaluation_exact(mdp, pi):
    '''Solve the linear system (16.14): U = R_pi + gamma P_pi U  (terminal utilities fixed at 0).'''
    n = mdp.nS
    P_pi = mdp.P[np.arange(n), pi]                       # n x n
    R_pi = np.einsum('ij,ij->i', P_pi, mdp.R[np.arange(n), pi])
    A = np.eye(n) - mdp.gamma * P_pi
    A[mdp.is_terminal] = 0; A[mdp.is_terminal, mdp.is_terminal] = 1; R_pi[mdp.is_terminal] = 0
    return np.linalg.solve(A, R_pi)

m = GridMDP(gamma=0.9)
U_true = value_iteration(m, 1e-14)
_, H9 = value_iteration(m, 1e-14, history=True)
iters = np.arange(15)
max_err  = [np.abs(H9[i] - U_true).max() for i in iters]
pol_loss = [np.abs(policy_evaluation_exact(m, greedy_policy(m, H9[i])[0]) - U_true).max() for i in iters]

plt.figure(figsize=(7, 4))
plt.plot(iters, max_err, 'o-', label='Max error ‖U_i − U‖')
plt.plot(iters, pol_loss, 's-', label='Policy loss ‖U^{π_i} − U‖')
plt.xlabel('Number of iterations'); plt.ylabel('Max error / Policy loss')
plt.title('γ = 0.9 (Fig. 16.8)'); plt.legend(); plt.grid(alpha=.3); plt.show()
first_opt = next(i for i in iters if pol_loss[i] < 1e-9)
print(f'π_i first optimal at i = {first_opt}; max utility error there = {max_err[first_opt]:.2f}')
# The book reports i = 5 (error 0.51); the exact index depends on conventions (e.g. what counts as
# iteration 0, how ties are broken), but the qualitative point is the same.

### 7.5 The stopping rule (Eq. 16.12) actually delivers the promised accuracy

In [ ]:
for eps in [1e-1, 1e-2, 1e-3, 1e-4]:
    U_eps, Hh = value_iteration(m, eps=eps, history=True)
    print(f'ε = {eps:g}: stopped after {len(Hh)-1:3d} iterations, true error = {np.abs(U_eps - U_true).max():.2e} < ε')

## 8. Policy iteration (§16.2.2, Fig. 16.9)

Alternate **exact policy evaluation** (solve $n$ linear equations, $O(n^3)$) and **policy improvement** (one-step look-ahead). Since there are finitely many policies and each step improves, it terminates — typically in a handful of iterations.

**Caveat with $\gamma=1$:** if the initial random policy is *improper* (e.g. walks into a wall forever), its utility is $-\infty$ and the linear system is singular. We therefore start from a random **proper** policy (one that reaches a terminal state with probability 1), or use $\gamma<1$.

In [ ]:
def is_proper(mdp, pi):
    try:
        U = policy_evaluation_exact(mdp, pi); return np.all(np.isfinite(U))
    except np.linalg.LinAlgError:
        return False

def policy_iteration(mdp, pi=None, verbose=False):
    if pi is None:
        while True:
            pi = rng.integers(mdp.nA, size=mdp.nS)
            if mdp.gamma < 1 or is_proper(mdp, pi): break
    pi = pi.copy(); k = 0
    while True:
        U = policy_evaluation_exact(mdp, pi)
        Q = mdp.q_values(U)
        a_star = Q.argmax(axis=1)
        better = (Q[np.arange(mdp.nS), a_star] > Q[np.arange(mdp.nS), pi] + 1e-12) & ~mdp.is_terminal
        k += 1
        if verbose:
            print(f'iter {k}: U(1,1) = {U[mdp.idx[(1,1)]]:.4f}, states changed = {better.sum()}')
        if not better.any():
            return pi, U, k
        pi[better] = a_star[better]

pi_pi, U_pi, k = policy_iteration(mdp, verbose=True)
print(f'\nPolicy iteration converged in {k} iterations;  max |U_PI − U_VI| = {np.abs(U_pi - U_star).max():.2e}')

# The simplified (linear) Bellman equations for the policy of Fig. 16.2(a), e.g. U(1,1):
P_pi = mdp.P[mdp.idx[(1,1)], mdp.ACTIONS.index('Up')]
print('Under π(1,1)=Up, successor distribution:', {mdp.states[j]: p for j, p in enumerate(P_pi) if p})

### Modified policy iteration
Replace the exact solve by $k$ sweeps of the *simplified* Bellman update $U\leftarrow \sum_{s'}P(s'|s,\pi(s))[R+\gamma U(s')]$. With $k=1$ this is value iteration; with $k\to\infty$ it is policy iteration.

In [ ]:
def modified_policy_iteration(mdp, k_eval, tol=1e-10, max_outer=10_000):
    U = np.zeros(mdp.nS); n = mdp.nS; backups = 0
    pi = mdp.q_values(U).argmax(axis=1)
    for outer in range(max_outer):
        P_pi = mdp.P[np.arange(n), pi]; R_pi = np.einsum('ij,ij->i', P_pi, mdp.R[np.arange(n), pi])
        for _ in range(k_eval):
            U = R_pi + mdp.gamma * P_pi @ U; U[mdp.is_terminal] = 0; backups += 1
        Q = mdp.q_values(U); U_new = Q.max(axis=1); U_new[mdp.is_terminal] = 0; backups += 1
        pi = Q.argmax(axis=1)
        if np.abs(U_new - U).max() < tol: return U_new, outer + 1, backups
        U = U_new
    return U, max_outer, backups

m = GridMDP(gamma=0.9)
for k_eval in [0, 1, 3, 10, 30]:      # k = 0 is plain value iteration
    U_m, outer, backups = modified_policy_iteration(m, k_eval)
    print(f'k = {k_eval:2d}: {outer:4d} improvement steps, {backups:4d} sweeps, '
          f'error {np.abs(U_m - U_true).max():.1e}')

### Asynchronous updates
Updating states one at a time, in place (Gauss–Seidel), using the newest values of neighbors, also converges — and usually faster. Ordering states from the goal outward helps even more.

In [ ]:
def async_vi(mdp, order, tol=1e-10):
    U = np.zeros(mdp.nS); sweeps = 0
    while True:
        delta = 0
        for i in order:
            if mdp.is_terminal[i]: continue
            new = (mdp.P[i] * (mdp.R[i] + mdp.gamma * U)).sum(axis=1).max()
            delta = max(delta, abs(new - U[i])); U[i] = new
        sweeps += 1
        if delta < tol: return U, sweeps

dist = np.array([abs(3 - row) + abs(4 - col) for (row, col) in m.states])   # Manhattan distance to (3,4)
_, H_sync = value_iteration(m, eps=1e-10, history=True)
for name, order in [('synchronous (book)', None), ('in-place, goal → far', np.argsort(dist)),
                    ('in-place, far → goal', np.argsort(-dist))]:
    if order is None:
        print(f'{name:22s}: {len(H_sync)-1} sweeps')
    else:
        U_a, sw = async_vi(m, order); print(f'{name:22s}: {sw} sweeps, error {np.abs(U_a-U_true).max():.1e}')

## 9. Linear programming (§16.2.3)

$$\min_U \sum_s U(s)\quad\text{s.t.}\quad U(s)\ge\sum_{s'}P(s'|s,a)[R(s,a,s')+\gamma U(s')]\ \ \forall s,a.$$

The optimal utilities are the *smallest* vector satisfying all the "≥ Bellman" constraints.

In [ ]:
from scipy.optimize import linprog

def solve_lp(mdp):
    A_ub, b_ub = [], []
    for i in range(mdp.nS):
        if mdp.is_terminal[i]: continue
        for a in range(mdp.nA):
            # sum_s' P (R + γU(s')) - U(s) <= 0   ⇔   (γP - e_i)·U <= -P·R
            row = mdp.gamma * mdp.P[i, a].copy(); row[i] -= 1
            A_ub.append(row); b_ub.append(-(mdp.P[i, a] * mdp.R[i, a]).sum())
    bounds = [(0, 0) if t else (None, None) for t in mdp.is_terminal]
    res = linprog(c=np.ones(mdp.nS), A_ub=np.array(A_ub), b_ub=np.array(b_ub), bounds=bounds, method='highs')
    return res.x, res

for g in [1.0, 0.9]:
    mm = GridMDP(gamma=g)
    U_lp, res = solve_lp(mm)
    print(f'γ = {g}: LP status = {res.message!r}; max |U_LP − U_VI| = '
          f'{np.abs(U_lp - value_iteration(mm, 1e-12)).max():.2e}')

## 10. Online algorithms (§16.2.4)

### 10.1 Expectimax and the $\epsilon$-horizon

A depth-$H$ expectimax tree (max nodes for the agent, chance nodes for nature) gives utilities within $\epsilon$ of exact if
$H=\lceil\log_\gamma(\epsilon(1-\gamma)/R_{\max})\rceil$.

In [ ]:
def eps_horizon(gamma, eps, Rmax=1.0):
    return int(np.ceil(np.log(eps * (1 - gamma) / Rmax) / np.log(gamma)))
for g in [0.5, 0.9, 0.99]:
    print(f'γ = {g}, ε = 0.1  →  H = {eps_horizon(g, 0.1)}')

In [ ]:
def expectimax(mdp, s, depth, leaf=lambda s: 0.0):
    '''Returns (value, best action). Leaves get value from `leaf` (an evaluation function).'''
    if mdp.is_terminal[s]: return 0.0, None
    if depth == 0: return leaf(s), None
    best, best_a = -np.inf, None
    for a in range(mdp.nA):
        v = sum(p * (mdp.R[s, a, s2] + mdp.gamma * expectimax(mdp, s2, depth - 1, leaf)[0])
                for s2, p in enumerate(mdp.P[s, a]) if p > 0)
        if v > best: best, best_a = v, a
    return best, best_a

m = GridMDP(gamma=0.9); U_true = value_iteration(m, 1e-12); pi_true, _ = greedy_policy(m, U_true)
s0 = m.idx[(1, 1)]
print(f'True U(1,1) at γ=0.9: {U_true[s0]:.4f}')
for d in [1, 2, 3, 4, 5, 6, 7]:
    v, a = expectimax(m, s0, d)
    print(f'depth {d}: value {v:+.4f}, action {m.ACTIONS[a]}')

Exhaustive expectimax grows as $(|A|\cdot b)^H$ — note how depth 7 already takes noticeably longer. Two remedies from the text:

1. **Sampling the chance nodes**: approximate $\sum_x P(x)f(x)\approx\frac1N\sum_i f(x_i)$.
2. **Treat the tree as a graph** ⇒ real-time dynamic programming (RTDP) — the explored states form a sub-MDP solved with Bellman updates. Below we compare plain expectimax with the sampled version.

In [ ]:
def sampled_expectimax(mdp, s, depth, n_samples, rng):
    if mdp.is_terminal[s] or depth == 0: return 0.0, None
    best, best_a = -np.inf, None
    for a in range(mdp.nA):
        succ = rng.choice(mdp.nS, size=n_samples, p=mdp.P[s, a])
        v = np.mean([mdp.R[s, a, s2] + mdp.gamma * sampled_expectimax(mdp, s2, depth - 1, n_samples, rng)[0]
                     for s2 in succ])
        if v > best: best, best_a = v, a
    return best, best_a

import time
# From (1,3) both exits are within reach, so the chance nodes really matter.
s13, depth = m.idx[(1, 3)], 4
t = time.time(); v, a = expectimax(m, s13, depth)
print(f'(1,3), exact depth {depth}:      value {v:+.4f}, action {m.ACTIONS[a]}  ({time.time()-t:.2f}s)')
for n_samp in [1, 2, 3, 5]:
    vals = [sampled_expectimax(m, s13, depth, n_samp, rng)[0] for _ in range(5)]
    print(f'N = {n_samp} samples/chance node: value {np.mean(vals):+.4f} ± {np.std(vals):.4f} (5 runs)')

With few samples the estimates are noisy **and biased upward**: the max node picks whichever action got lucky samples. Both effects shrink as $N$ grows.

### 10.2 UCT on the 4×3 world (Fig. 16.11)

UCT (Monte Carlo tree search with the UCB1 selection rule) adapted to MDPs: nature's moves are sampled from $P$, and we back up discounted rewards instead of win/loss.
At every real step, the agent builds a fresh tree of $n$ playouts from its current state (random playout policy), then executes the action with the best mean value. We measure the average total reward from $(1,1)$ with $\gamma=1$.

The book's point: UCT is **not** impressive here — around 160 playouts per move gets about $0.4$, while the optimal expected return is $0.7453$. The world is "loopy" and a tree (rather than a graph) wastes effort re-discovering the same states.

In [ ]:
class Node:
    __slots__ = ('N', 'Na', 'Wa', 'children')
    def __init__(self, nA):
        self.N, self.Na, self.Wa, self.children = 0, np.zeros(nA), np.zeros(nA), {}

def uct_choose(mdp, s_root, n_playouts, rng, c=1.4, max_depth=60, rollout_len=60):
    root = Node(mdp.nA)
    for _ in range(n_playouts):
        node, s, path, G_steps = root, s_root, [], []
        depth = 0
        # --- selection / expansion ---
        while not mdp.is_terminal[s] and depth < max_depth:
            untried = np.flatnonzero(node.Na == 0)
            if len(untried):
                a = rng.choice(untried)
            else:
                ucb = node.Wa / node.Na + c * np.sqrt(np.log(node.N) / node.Na)
                a = int(np.argmax(ucb))
            s2, rew = mdp.sample(s, a, rng)
            path.append((node, a)); G_steps.append(rew)
            depth += 1
            key = (a, s2)
            if key not in node.children:          # expand one new node, then roll out
                node.children[key] = Node(mdp.nA); node, s = node.children[key], s2
                break
            node, s = node.children[key], s2
        # --- random playout ---
        rollout = 0.0
        for _ in range(rollout_len):
            if mdp.is_terminal[s]: break
            s, rew = mdp.sample(s, rng.integers(mdp.nA), rng); rollout += rew
        # --- backup (γ = 1: plain sums) ---
        G = rollout
        for (nd, a), rew in zip(reversed(path), reversed(G_steps)):
            G = rew + mdp.gamma * G
            nd.N += 1; nd.Na[a] += 1; nd.Wa[a] += G
    visited = root.Na > 0
    means = np.where(visited, root.Wa / np.maximum(root.Na, 1), -np.inf)
    return int(np.argmax(means))

def uct_episode(mdp, start, n_playouts, rng, max_steps=200):
    s, total = mdp.idx[start], 0.0
    for _ in range(max_steps):
        if mdp.is_terminal[s]: break
        a = uct_choose(mdp, s, n_playouts, rng)
        s, rew = mdp.sample(s, a, rng); total += rew
    return total

In [ ]:
# Fewer runs than the book's 1000 per point to keep this quick; increase n_runs for smoother curves.
playouts = [5, 10, 20, 40, 80, 160]
n_runs = 100
uct_rng = np.random.default_rng(1)
avg = []
t0 = time.time()
for n in playouts:
    avg.append(np.mean([uct_episode(mdp, (1, 1), n, uct_rng) for _ in range(n_runs)]))
    print(f'{n:4d} playouts/move: average total reward = {avg[-1]:+.3f}   ({time.time()-t0:.0f}s elapsed)')

plt.figure(figsize=(7, 4))
plt.plot(playouts, avg, 'o-', label='UCT (random playouts)')
plt.axhline(U_star[mdp.idx[(1, 1)]], ls='--', c='k', label='optimal U(1,1) = 0.7453')
plt.xlabel('Number of playouts'); plt.ylabel('Average total reward')
plt.title(f'UCT on the 4×3 world, γ = 1 ({n_runs} runs/point) — cf. Fig. 16.11'); plt.legend(); plt.grid(alpha=.3)
plt.show()

## 11. Summary

| Method | What it computes | Cost on this problem | Notes |
|---|---|---|---|
| Value iteration | $U$ via $U\leftarrow BU$ | dozens of sweeps ($\gamma=0.9$); more as $\gamma\to1$ | contraction with factor $\gamma$; policy optimal long before $U$ converges |
| Policy iteration | $\pi^*$ via eval/improve | 3–6 iterations, each an $O(n^3)$ solve | needs proper policies when $\gamma=1$ |
| Modified PI | between VI and PI | tunable $k$ | usually the practical sweet spot |
| LP | $U$ as smallest feasible vector | one LP | polynomial time, rarely fastest in practice |
| Expectimax / UCT | decision at the current state only | grows with depth / playouts | online; poor on "loopy" worlds like 4×3 |

**Exercises to try**
1. Change `p_intended` to 0.9 or 1.0 — when does the tie at $(1,3)$ disappear?
2. Find the exact $r$ thresholds of Fig. 16.2(b) by bisection rather than a sweep.
3. Use a *non-zero* $\Phi$ at terminal states in §6 and observe the shaping theorem fail.
4. Replace the random rollout in UCT with the greedy policy w.r.t. a crude heuristic (e.g. negative Manhattan distance to $(3,4)$). How many playouts are now needed to reach 0.7?